# 第 16 课：流式编码器——因果卷积、缓存与 Chunk Attention

CTC head 可以逐帧输出，但编码器若查看无限未来，系统仍不流式。本课用数值实验验证“离线与分块结果一致”。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 流式 ASR |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 15 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | 因果卷积、encoder cache、chunk attention |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：因果卷积、encoder cache、chunk attention。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

import torch
import torch.nn as nn
import torch.nn.functional as F
from ipywidgets import interact, IntSlider
torch.manual_seed(1)

项目根目录: <REPO_ROOT>


## 1. 因果卷积只看当前和过去

In [2]:
conv=nn.Conv1d(3,5,kernel_size=3,bias=False)
x=torch.randn(1,3,17)
offline=conv(F.pad(x,(2,0)))

def stream_conv(x,chunk_size):
    cache=torch.zeros(x.size(0),x.size(1),2); outputs=[]
    for s in range(0,x.size(-1),chunk_size):
        chunk=x[:,:,s:s+chunk_size]; joined=torch.cat([cache,chunk],dim=-1)
        outputs.append(conv(joined)); cache=joined[:,:,-2:]
    return torch.cat(outputs,dim=-1)

for cs in [1,2,4,7,20]: print(cs,(stream_conv(x,cs)-offline).abs().max().item())

1 1.1920928955078125e-07
2 1.1920928955078125e-07
4 1.1920928955078125e-07
7 1.1920928955078125e-07
20 0.0


缓存的是输入历史，不是把整段过去重新计算。kernel=3、dilation=1 时需要 2 个历史位置；多层卷积要分别维护各层缓存。

## 2. Chunk Attention mask

严格 causal attention 只能看左侧；chunk attention 允许当前块内部互相看，并可保留有限或无限左上下文。

In [3]:
def chunk_mask(T,chunk,left_chunks):
    mask=np.zeros((T,T),dtype=int)
    for q in range(T):
        current=q//chunk; first=max(0,current-left_chunks); lo=first*chunk; hi=min(T,(current+1)*chunk)
        mask[q,lo:hi]=1
    return mask

@interact(chunk=IntSlider(min=1,max=8,value=4),left_chunks=IntSlider(min=0,max=4,value=1))
def show_mask(chunk=4,left_chunks=1):
    m=chunk_mask(24,chunk,left_chunks)
    plt.imshow(m,origin="lower",cmap="Blues",vmin=0,vmax=1)
    plt.xlabel("Key frame");plt.ylabel("Query frame");plt.title("Allowed attention positions");plt.show()

interactive(children=(IntSlider(value=4, description='chunk', max=8, min=1), IntSlider(value=1, description='l…

## 3. 右上下文与算法延迟

允许当前帧查看未来 $R$ 帧，帧移 10 ms，则至少增加约 $R×10$ ms 等待。chunk attention 常常还要先收满一个块。准确率、吞吐量和延迟要联合衡量。

In [4]:
hop_ms=10
for right in [0,4,8,16,32]: print(f"right context={right:2d} frames -> about {right*hop_ms:3d} ms lookahead")

right context= 0 frames -> about   0 ms lookahead
right context= 4 frames -> about  40 ms lookahead
right context= 8 frames -> about  80 ms lookahead
right context=16 frames -> about 160 ms lookahead
right context=32 frames -> about 320 ms lookahead


## 本课测试

1. CTC + 双向 LSTM 是否严格流式？
2. causal conv 的左 padding 与普通 same padding有什么区别？
3. cache 的主要工程价值是什么？
4. chunk 越大通常对上下文和延迟各有什么影响？
5. 流式结果与离线结果必须完全相同吗？

<details><summary>展开参考答案</summary>

1. 否。2. causal 只在左侧补，不引入未来。3. 复用历史计算。4. 上下文通常更充分，但等待和单块计算可能增加。5. 不一定；若离线模型使用更多未来信息，两者可能不同，应分别评估。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 16 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `因果卷积`、`encoder cache`、`chunk attention`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**普通 same padding 偷看未来**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**比较多种 chunk 下离线/在线最大误差**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**把右上下文换算为算法延迟**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：因果卷积、encoder cache、chunk attention。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 因果卷积、encoder cache、chunk attention。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
